# 29F — Source-balance candidate pose audit (CPU)

Run all on **CPU**. The notebook downloads only pinned manifests and the five candidate 360p source videos if they are not already cached by 29D/29E in the same Colab session. It extracts 15 FPS MediaPipe pose for broad candidate windows and saves pose-overlay sheets plus quality metrics in one ZIP. This is **draft evidence only**: no movement labels are approved and no output is training eligible. A YouTube access failure is recorded per source; other sources continue. No raw video is exported.


In [ ]:
%pip install -q --upgrade "yt-dlp[default]" mediapipe==0.10.35 opencv-python-headless pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib,json,platform,re,shutil,subprocess,urllib.request,zipfile
import cv2,mediapipe as mp,numpy as np,pandas as pd
from google.colab import files
from IPython.display import display

PINNED_COMMIT='5315822130c3bbd0933f82508b14a1f75fc683ea'
RAW=f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}/'
INPUTS={
 'ml/data/source_balance_v1/candidates.csv':'06cc829a21f7d91a02bea5913cabad06796050ec316a3037e5a829bc18e88cea',
 'ml/data/source_balance_v1/dense_visual_first_pass.csv':'6a9fccdb444edec48287d713832aee8c87a524a693696d0796d5f7b2e9fa71bd',
 'ml/data/youtube_creator_benchmark_v1/candidates.csv':'aa3063810fa3b0e9fe169c04281ef65cc33a442d18be0fa0e5fec7669e06268b',
 'ml/data/youtube_creator_benchmark_v1/excluded_source_inventory.csv':'ecf18c29ef5b7ba1eca3f45677e463df256b965888f3913ae55606807baf2232',
 'ml/data/youtube_mixed_workouts_v1/manifest.csv':'1d58ed25261a172e49abe3f9b7313236d339e5fbe067ecccf838c6f187efb3eb',
}
WORK=Path('/content/calorie_source_balance_v1_work')
OUTPUT=Path('/content/calorie_source_balance_v1_pose_audit')
if OUTPUT.exists():shutil.rmtree(OUTPUT)
for folder in (WORK,WORK/'videos',WORK/'pose_segments',OUTPUT,OUTPUT/'continuous_landmarks',OUTPUT/'pose_contact_sheets',OUTPUT/'metadata'):
 folder.mkdir(parents=True,exist_ok=True)

def sha256(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):d.update(block)
 return d.hexdigest()

def fetch(relative,digest):
 path=WORK/'pinned_inputs'/relative
 path.parent.mkdir(parents=True,exist_ok=True)
 if not path.exists() or sha256(path)!=digest:
  tmp=path.with_suffix(path.suffix+'.part');tmp.unlink(missing_ok=True)
  urllib.request.urlretrieve(RAW+relative,tmp)
  assert sha256(tmp)==digest,relative
  tmp.replace(path)
 return path

paths={name:fetch(name,digest) for name,digest in INPUTS.items()}
assert len(set(paths.values()))==len(INPUTS)
assert all(sha256(paths[name])==digest for name,digest in INPUTS.items())
candidates=pd.read_csv(paths['ml/data/source_balance_v1/candidates.csv'])
triage=pd.read_csv(paths['ml/data/source_balance_v1/dense_visual_first_pass.csv']).fillna('')
reserved=pd.concat([pd.read_csv(paths[name]) for name in (
 'ml/data/youtube_creator_benchmark_v1/candidates.csv',
 'ml/data/youtube_creator_benchmark_v1/excluded_source_inventory.csv',
 'ml/data/youtube_mixed_workouts_v1/manifest.csv')],ignore_index=True)
normalize=lambda value:re.sub('[^a-z0-9]+','',str(value).casefold())
assert candidates.video_id.is_unique and candidates.creator.map(normalize).is_unique
assert set(candidates.video_id).isdisjoint(set(reserved.video_id))
assert set(candidates.creator.map(normalize)).isdisjoint(set(reserved.creator.map(normalize)))
assert (candidates.status=='metadata_only').all() and (~candidates.training_allowed).all()
assert triage.review_id.is_unique and set(triage.video_id).issubset(set(candidates.video_id))
assert (triage.review_status=='draft').all() and (~triage.training_allowed).all()
assert triage.needs_second_review.all()
chosen=triage[triage.first_pass_status.isin(['pose_review','second_review'])].copy()
windows=[]
for row in chosen.itertuples(index=False):
 for j,pair in enumerate(row.approx_candidate_windows_s.split('|')):
  start,end=map(float,pair.split('-'))
  assert float(row.source_start_s)<=start<end<=float(row.source_end_s),(row.review_id,pair)
  windows.append({'window_id':f'{row.review_id}_{j+1:02d}','review_id':row.review_id,
   'video_id':row.video_id,'target_hypothesis':row.target_hypothesis,
   'start_s':start,'end_s':end,'first_pass_status':row.first_pass_status})
windows=pd.DataFrame(windows)
assert windows.window_id.is_unique and len(windows)>0
selected=candidates[candidates.video_id.isin(chosen.video_id)].reset_index(drop=True)
assert len(selected)==5 and len(windows)==sum(chosen.approx_candidate_windows_s.map(lambda value:len(value.split('|'))))
print('CPU pose audit:',len(windows),'broad windows across',len(selected),'sources; training remains forbidden')
display(windows.groupby(['video_id','target_hypothesis']).size().rename('windows').reset_index())


In [ ]:
def deno_args():
 asset={'x86_64':'deno-x86_64-unknown-linux-gnu.zip','aarch64':'deno-aarch64-unknown-linux-gnu.zip'}.get(platform.machine().lower())
 if not asset:return []
 deno=WORK/'deno'/'deno'
 try:
  if not deno.exists():
   deno.parent.mkdir(parents=True,exist_ok=True)
   archive=deno.parent/'deno.zip'
   urllib.request.urlretrieve('https://github.com/denoland/deno/releases/latest/download/'+asset,archive)
   with zipfile.ZipFile(archive) as z:z.extract('deno',deno.parent)
   deno.chmod(0o755)
  subprocess.run([str(deno),'--version'],check=True,stdout=subprocess.DEVNULL,timeout=10)
  return ['--js-runtimes',f'deno:{deno}','--remote-components','ejs:github']
 except Exception as exc:
  print('Deno unavailable; yt-dlp fallback:',type(exc).__name__)
  return []
JS_ARGS=deno_args()

def run(args,timeout=300):
 try:
  p=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout,check=False)
  return p.returncode,p.stdout
 except subprocess.TimeoutExpired as exc:return 124,str(exc)

def info(path):
 code,out=run(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=width,height,duration','-of','json',str(path)],30)
 if code:return None
 try:
  s=json.loads(out)['streams'][0]
  return {'width':int(s['width']),'height':int(s['height']),'duration_s':float(s.get('duration') or 0)}
 except (ValueError,IndexError,KeyError):return None

def obtain(row):
 cached=[p for p in (WORK/'videos').glob(row.video_id+'.*') if p.suffix in {'.mp4','.mkv','.webm'} and info(p) and info(p)['height']==360]
 if cached:return cached[0],''
 failures=[]
 for fmt in ('18/best[height=360][ext=mp4]','bestvideo[height=360]+bestaudio/best[height=360]'):
  args=['yt-dlp','--no-playlist','--no-progress','--retries','2','--fragment-retries','2','--socket-timeout','20','-f',fmt,'--merge-output-format','mp4','-o',str(WORK/'videos'/(row.video_id+'.%(ext)s')),*JS_ARGS,row.source_url]
  code,out=run(args,300)
  matches=[p for p in (WORK/'videos').glob(row.video_id+'.*') if p.suffix in {'.mp4','.mkv','.webm'} and info(p) and info(p)['height']==360]
  if code==0 and matches:return matches[0],''
  failures.append(' '.join(out.splitlines()[-4:])[-500:])
  if 'Sign in to confirm' in out or 'not a bot' in out:break
 return None,' | '.join(failures)

source_paths={}
source_status=[]
reserved_channels=set(reserved.creator.map(normalize))
for row in selected.itertuples(index=False):
 record={'video_id':row.video_id,'creator':row.creator,'status':'unavailable','error':''}
 print('Getting',row.video_id,row.creator)
 video,error=obtain(row)
 if video is None:
  record['error']=error;source_status.append(record);continue
 metadata_path=OUTPUT/'metadata'/(row.video_id+'.json')
 code,out=run(['yt-dlp','--skip-download','--dump-single-json','--no-warnings',*JS_ARGS,row.source_url],90)
 if code==0:
  try:
   raw=json.loads(out)
   metadata={key:raw.get(key) for key in ('id','title','channel','channel_id','duration','license','availability','webpage_url')}
   metadata_path.write_text(json.dumps(metadata,indent=2,ensure_ascii=False)+'\n')
   if metadata['id']!=row.video_id or normalize(metadata['channel']) in reserved_channels:
    record.update(status='identity_conflict',error='Video or observed channel conflicts with reserved sources')
    source_status.append(record);continue
  except (ValueError,TypeError):
   record.update(status='metadata_review',error='Could not parse observed channel metadata')
   source_status.append(record);continue
 source_paths[row.video_id]=video
 record['status']='ready';source_status.append(record)

print('Video sources ready:',len(source_paths),'of',len(selected))


In [ ]:
MODEL=WORK/'pose_landmarker_full.task'
MODEL_URL=('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
 'pose_landmarker_full/float16/latest/pose_landmarker_full.task')
if not MODEL.exists():urllib.request.urlretrieve(MODEL_URL,MODEL)
assert MODEL.stat().st_size>1_000_000
TARGET_FPS=15
KEEP_17=[0,2,5,7,8,11,12,13,14,15,16,23,24,25,26,27,28]
SKELETON=[(11,13),(13,15),(12,14),(14,16),(11,12),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12)]

def normalize_pose(frame):
 xyz=frame[:,:3].copy();hip=(xyz[11]+xyz[12])/2;shoulder=(xyz[5]+xyz[6])/2
 scale=max(float(np.linalg.norm(xyz[5]-xyz[6])),float(np.linalg.norm(shoulder-hip)),1e-6)
 return np.concatenate([(xyz-hip)/scale,frame[:,3:4]],axis=1)

def segment_video(source,start,end,window_id):
 target=WORK/'pose_segments'/(window_id+'.mp4')
 if target.exists() and info(target) and info(target)['duration_s']>=end-start-0.6:return target
 target.unlink(missing_ok=True)
 args=['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(start),'-i',str(source),
  '-t',str(end-start),'-map','0:v:0','-vf','fps=15,scale=-2:360','-c:v','libx264',
  '-preset','ultrafast','-crf','20','-pix_fmt','yuv420p','-an',str(target)]
 code,out=run(args,600)
 if code or not info(target):raise RuntimeError('segment decode: '+out[-500:])
 return target

def overlay(frame,points,timestamp):
 tile=cv2.resize(frame,(320,180));h,w=frame.shape[:2]
 if points is not None:
  for a,b in SKELETON:
   x1,y1=int(points[a].x*320),int(points[a].y*180)
   x2,y2=int(points[b].x*320),int(points[b].y*180)
   if min(points[a].visibility,points[b].visibility)>=.5:cv2.line(tile,(x1,y1),(x2,y2),(0,255,0),2)
  for p in points:
   if p.visibility>=.5:cv2.circle(tile,(int(p.x*320),int(p.y*180)),2,(0,0,255),-1)
 cv2.rectangle(tile,(0,0),(94,24),(0,0,0),-1)
 cv2.putText(tile,f'{int(timestamp//60):02d}:{timestamp%60:04.1f}',(4,17),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,255,255),1)
 return tile

def pose_window(row,video):
 segment=segment_video(video,float(row.start_s),float(row.end_s),row.window_id)
 cap=cv2.VideoCapture(str(segment))
 if not cap.isOpened():raise RuntimeError('OpenCV could not open segment')
 source_fps=float(cap.get(cv2.CAP_PROP_FPS) or TARGET_FPS)
 options=mp.tasks.vision.PoseLandmarkerOptions(
  base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),
  running_mode=mp.tasks.vision.RunningMode.VIDEO,num_poses=1,
  min_pose_detection_confidence=.5,min_pose_presence_confidence=.5,
  min_tracking_confidence=.5,output_segmentation_masks=False)
 raw=[];normalized=[];times=[];detected=[];preview=[];i=0
 try:
  with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
   while True:
    ok,frame=cap.read()
    if not ok:break
    time_s=float(row.start_s)+i/source_fps
    result=landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)),int(round(i/source_fps*1000)))
    points=result.pose_landmarks[0] if result.pose_landmarks else None
    if points:
     full=np.asarray([[p.x,p.y,p.z,float(p.visibility or 0)] for p in points],dtype=np.float32)
     raw.append(full);normalized.append(normalize_pose(full[KEEP_17]));detected.append(True)
    else:
     raw.append(np.full((33,4),np.nan,np.float32));normalized.append(np.full((17,4),np.nan,np.float32));detected.append(False)
    times.append(time_s)
    if i%max(1,round(source_fps))==0:preview.append(overlay(frame,points,time_s))
    i+=1
 finally:cap.release()
 if len(times)<max(15,int((row.end_s-row.start_s)*TARGET_FPS*.8)):
  raise RuntimeError(f'Only {len(times)} decoded pose frames')
 raw=np.stack(raw);normalized=np.stack(normalized);detected=np.asarray(detected,dtype=bool)
 output=OUTPUT/'continuous_landmarks'/(row.window_id+'.npz')
 np.savez_compressed(output,image_landmarks=raw,landmarks=normalized,
  timestamps_s=np.asarray(times,np.float32),detected=detected,
  source_fps=np.float32(source_fps),target_fps=np.float32(TARGET_FPS),
  video_id=row.video_id,group_id=f'youtube:{row.video_id}',
  target_hypothesis=row.target_hypothesis,training_allowed=False)
 page_root=OUTPUT/'pose_contact_sheets'/row.window_id;page_root.mkdir(parents=True,exist_ok=True)
 for page_idx,start_idx in enumerate(range(0,len(preview),12)):
  tiles=preview[start_idx:start_idx+12]
  while len(tiles)<12:tiles.append(np.zeros_like(tiles[0]))
  page=np.vstack([np.hstack(tiles[j:j+4]) for j in (0,4,8)])
  cv2.imwrite(str(page_root/f'{page_idx:03d}.jpg'),page,[cv2.IMWRITE_JPEG_QUALITY,85])
 vis=raw[:,:,3]
 coverage=float(np.nanmean(vis>=.5)) if detected.any() else 0.
 return {'frames':len(times),'detection_rate':float(detected.mean()),'visible_joint_rate':coverage,
  'pose_pages':len(list(page_root.glob('*.jpg'))),'landmark_path':str(output.relative_to(OUTPUT))}

rows=[]
for row in windows.itertuples(index=False):
 record=row._asdict();record.update(status='source_unavailable',frames=0,detection_rate=np.nan,
  visible_joint_rate=np.nan,pose_pages=0,landmark_path='',error='')
 if row.video_id in source_paths:
  try:
   if float(row.end_s)>info(source_paths[row.video_id])['duration_s']:raise RuntimeError('Video shorter than candidate window')
   record.update(pose_window(row,source_paths[row.video_id]),status='pose_ready')
  except Exception as exc:record.update(status='pose_failed',error=f'{type(exc).__name__}: {exc}'[:500])
 else:record['error']='See source_status.csv'
 rows.append(record)
 print(row.window_id,record['status'],record['frames'],record['detection_rate'])
quality=pd.DataFrame(rows)
quality['review_status']='draft';quality['needs_second_review']=True;quality['training_allowed']=False
quality.to_csv(OUTPUT/'pose_quality.csv',index=False)
pd.DataFrame(source_status).to_csv(OUTPUT/'source_status.csv',index=False)
display(quality[['window_id','status','frames','detection_rate','visible_joint_rate']])


Pose metrics and overlay sheets are quality evidence, not class labels. Inspect actor identity, occlusion, and exact action boundaries before annotation. Keep all windows from a source video in one split.


In [ ]:
shutil.copy2(paths['ml/data/source_balance_v1/candidates.csv'],OUTPUT/'candidates.csv')
shutil.copy2(paths['ml/data/source_balance_v1/dense_visual_first_pass.csv'],OUTPUT/'dense_visual_first_pass.csv')
windows.to_csv(OUTPUT/'candidate_windows.csv',index=False)
metadata={'notebook':'29f_colab_audit_source_balance_pose_candidates.ipynb','runtime':'CPU',
 'pinned_commit':PINNED_COMMIT,'target_fps':TARGET_FPS,'window_count':len(windows),
 'pose_ready':int((quality.status=='pose_ready').sum()),'training_allowed':False,
 'review_status':'draft','raw_video_exported':False,
 'next_step':'human pose audit and independent exact motion/boundary review before training'}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
checks=[]
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file() and p.name!='SHA256SUMS.txt'):
 checks.append(f'{sha256(path)}  {path.relative_to(OUTPUT)}')
(OUTPUT/'SHA256SUMS.txt').write_text('\n'.join(checks)+'\n')
assert not list(OUTPUT.rglob('*.mp4')) and not list(OUTPUT.rglob('*.webm'))
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
print('Created:',archive,round(Path(archive).stat().st_size/1e6,1),'MB')
files.download(archive)
